In [4]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/home/yuanshanwu/Documents/GitHub/QuantUS-Plugins-CEUS
/home/yuanshanwu/Documents/GitHub


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [5]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['mp4', 'custom_dicom', 'nifti', 'avi']


In [6]:
scan_type = 'nifti'

# Takes the DICOM file as input for contrast enhanced ultrasound (CEUS) scans
CEUS_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P07-V02-CE01/HighQuality/UCSD-P07-V02-CE01RUN2_09.39.44/UCSD-P07-V02-CE01RUN2_09.39.44_mf_sip_capture_50_2_1_0_CEUS.nii'
bmode_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P07-V02-CE01/HighQuality/UCSD-P07-V02-CE01RUN2_09.39.44/UCSD-P07-V02-CE01RUN2_09.39.44_mf_sip_capture_50_2_1_0_BMODE.nii'
scan_loader_kwargs = {
}

In [7]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, CEUS_scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

In [8]:
import pickle
with open('/home/yuanshanwu/Documents/TUL/CEUS-Studies/P07-V02-CE01/HighQuality/seg_data_with_mc_correlation_based.pkl', 'rb') as f:
    seg_data = pickle.load(f)

# CEUS Quantitative Temporal Curve Visualization

In [9]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))
print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis types: ['curves', 'curves_paramap']
Available analysis functions: ['pyradiomics', 'tic']


In [10]:
analysis_type = 'curves_paramap'
analysis_funcs = ['tic']

# Find all required kwargs for the analysis functions
analysis_funcs = analysis_funcs if len(analysis_funcs) else list(all_analysis_funcs[analysis_type].keys())
required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: ['cor_vox_len', 'sag_vox_ovrlp', 'ax_vox_len', 'sag_vox_len', 'cor_vox_ovrlp', 'ax_vox_ovrlp']


In [11]:
analysis_kwargs = {
    'ax_vox_ovrlp': 75,  # %
    'sag_vox_ovrlp': 75,  # %
    'cor_vox_ovrlp': 75,  # %
    'ax_vox_len': 5.0,  # mm
    'sag_vox_len': 2.5,  # mm
    'cor_vox_len': 2.5,  # mm
    'curves_output_path': '', # don't export the curves we generate
}

In [12]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

Computing curves:   0%|          | 0/530 [00:22<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# Save the segmentation data with motion compensation applied
import pickle
from pathlib import Path

# Build output path: split CEUS_scan_path at 'HighQuality' and append pkl filename
base_path = CEUS_scan_path.split('HighQuality')[0]
output_pkl_path = str(Path(base_path) / 'HighQuality' / 'ax_5_sag_2.5_cor_2.5_with_mc_correlation_based.pkl')
print(f"Saving to: {output_pkl_path}")
# Save seg_data with motion compensation
with open(output_pkl_path, 'wb') as f:
    pickle.dump(analysis_obj, f)

Saving to: /home/yuanshanwu/Documents/TUL/CEUS-Studies/P07-V02-CE01/HighQuality/ax_5_sag_2.5_cor_2.5_with_mc_correlation_based.pkl


## Curve Quantification for each sub-kernels

In [ ]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['auc_no_fit', 'cmus_firstorder', 'dte', 'first_order_full', 'first_order_select', 'lognormal_fit_full', 'lognormal_fit_select', 'wash_rates'])


In [ ]:
function_names = ['lognormal_fit_full'] # Empty list will use all functions
output_path = 'test_quants.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
}

In [ ]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

/home/yuanshanwu/Documents/GitHub/QuantUS-Plugins-CEUS/src/curve_quantification/transforms.py:7: RuntimeWarning: overflow encountered in divide
  result = (auc / (shifted * sigma * np.sqrt(2 * np.pi))) * np.exp(-((np.log(shifted) - mu) ** 2) / (2 * sigma ** 2))


## Visualize the parametric Map

In [13]:
from src.visualizations.options import get_visualization_types

types, funcs = get_visualization_types()
print("Available visualization types:", list(types.keys()))
print("Available visualization functions:", list(funcs.keys()))

Available visualization types: ['paramap']
Available visualization functions: ['paramap']


In [14]:
vis_type = 'paramap'
params = [] # Empty list will use all parameters 
vis_funcs = []
vis_kwargs = {
    'paramap_folder_path': 'paramaps_LOGNORMAL',
    'hide_all_visualizations': False,  # Set to True to hide all visualizations
}

In [ ]:
from src.entrypoints import visualization_step

vis_obj = visualization_step(curve_quant, vis_type, params, vis_funcs, **vis_kwargs)

In [ ]:
from src.map_showing import *

# 3. Overlay T0 on CEUS image (with motion compensation!)
viewer = view_T0_with_CEUS(vis_obj, image_data, 
                          param_name='T0_full_TIC',
                          time_point=0)  # Frame 50
napari.run()

In [ ]:
vis_obj.paramap_names

['AUC_full_TIC',
 'PE_full_TIC',
 'TP_full_TIC',
 'MTT_full_TIC',
 'T0_full_TIC',
 'Mu_full_TIC',
 'Sigma_full_TIC',
 'PE_Ix_full_TIC',
 'T0_map_pixel']

# Pixel-wise T0 map

In [15]:
from src.pixel_T0 import generate_t0_map_3d, get_t0_statistics_3d

t0_map = generate_t0_map_3d(
    image_data, seg_data,
    threshold=130, start_frame=0, end_frame=150,
    min_consecutive_frames=3
)
stats = get_t0_statistics_3d(t0_map, seg_data)

In [17]:
from src.map_showing import *

# 3. Overlay T0 on CEUS image (with motion compensation!)
viewer = view_heatmap(t0_map, image_data, 
                      time_point=0, seg_mask=seg_data.seg_mask)  #0
napari.run()

T0 Map: contrast=[0, 119.0], mean(activated)=75.3, activated voxels=6084
